# Extracting Camera Intrinsic

## The Intrinsic Matrix ($K$)

The data we are extracting forms a $3 \times 3$ matrix called the Camera Matrix ($K$), which OpenCV requires for the next step (the PnP math). It looks like this:
$$K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$$
$f_x, f_y$ (Focal Lengths): Represented in pixels. If the lens is perfectly symmetrical, $f_x$ and $f_y$ will be almost identical.
$c_x, c_y$ (Principal Point): The exact pixel coordinate where the optical axis intersects the image sensor. For your 1280x720 resolution (noted in your SAM3 notebook), this should be roughly at pixel (640, 360).

In [4]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pyrealsense2 as rs

WIDTH = 1280
HEIGHT = 720
FPS = 30
PREVIEW_SECONDS = None  # Set to a number to auto-stop, or leave as None and stop the cell manually.

ctx = rs.context()
devices = list(ctx.query_devices())
if not devices:
    raise RuntimeError(
        "No RealSense device is visible to librealsense in this Docker container. "
        "Passing only /dev/video* is usually not enough; expose the camera USB bus "
        "(for example /dev/bus/usb) and udev metadata to the container, then rerun this cell."
    )

device = devices[0]
device_name = device.get_info(rs.camera_info.name) if device.supports(rs.camera_info.name) else "RealSense camera"
serial_number = device.get_info(rs.camera_info.serial_number) if device.supports(rs.camera_info.serial_number) else None

pipeline = rs.pipeline()
config = rs.config()
if serial_number:
    config.enable_device(serial_number)
config.enable_stream(rs.stream.color, WIDTH, HEIGHT, rs.format.bgr8, FPS)

print(f"Starting live preview from {device_name}...")
pipeline_started = False
plt.ion()
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_title("RealSense live color preview")
ax.axis("off")
image_artist = None
start_time = time.time()

try:
    pipeline.start(config)
    pipeline_started = True

    # Let auto-exposure settle for a few frames before displaying.
    for _ in range(10):
        pipeline.wait_for_frames()

    while True:
        frames = pipeline.wait_for_frames()
        color_frame = frames.get_color_frame()
        if not color_frame:
            continue

        color_image = np.asanyarray(color_frame.get_data())[:, :, ::-1]

        if image_artist is None:
            image_artist = ax.imshow(color_image)
            plt.show(block=False)
        else:
            image_artist.set_data(color_image)

        fig.canvas.draw_idle()
        fig.canvas.flush_events()
        plt.pause(0.001)

        if PREVIEW_SECONDS is not None and (time.time() - start_time) >= PREVIEW_SECONDS:
            print("Preview finished.")
            break

except KeyboardInterrupt:
    print("Preview stopped by user.")
finally:
    if pipeline_started:
        pipeline.stop()
    plt.ioff()

RuntimeError: No RealSense device is visible to librealsense in this Docker container. Passing only /dev/video* is usually not enough; expose the camera USB bus (for example /dev/bus/usb) and udev metadata to the container, then rerun this cell.

In [5]:
import pyrealsense2 as rs
import numpy as np

WIDTH = 1280
HEIGHT = 720
FPS = 30

ctx = rs.context()
devices = list(ctx.query_devices())
if not devices:
    raise RuntimeError(
        "No RealSense device is visible to librealsense in this Docker container. "
        "If the camera is physically connected, recreate or reconfigure the container with USB passthrough "
        "(for example /dev/bus/usb and udev metadata), then rerun this cell."
    )

device = devices[0]
device_name = device.get_info(rs.camera_info.name) if device.supports(rs.camera_info.name) else "RealSense camera"
serial_number = device.get_info(rs.camera_info.serial_number) if device.supports(rs.camera_info.serial_number) else None

# 1. Initialize the RealSense pipeline
pipeline = rs.pipeline()
config = rs.config()
if serial_number:
    config.enable_device(serial_number)

# 2. Configure the stream to match your SAM3 dataset resolution
config.enable_stream(rs.stream.color, WIDTH, HEIGHT, rs.format.bgr8, FPS)

print(f"Starting RealSense pipeline for {device_name}...")
pipeline_started = False

try:
    profile = pipeline.start(config)
    pipeline_started = True

    # Let the stream warm up before reading intrinsics.
    for _ in range(10):
        pipeline.wait_for_frames()

    # 3. Get the color stream profile
    color_stream = profile.get_stream(rs.stream.color)
    color_profile = rs.video_stream_profile(color_stream)

    # 4. Extract the intrinsic parameters from the firmware
    intrinsics = color_profile.get_intrinsics()

    # 5. Format it into the standard OpenCV 3x3 K-Matrix
    K_matrix = np.array([
        [intrinsics.fx, 0,             intrinsics.ppx],
        [0,             intrinsics.fy, intrinsics.ppy],
        [0,             0,             1]
    ], dtype=float)

    distortion_coeffs = np.array(intrinsics.coeffs[:5], dtype=float)

    print("\n=== RealSense Factory Intrinsics ===")
    print(f"Device: {device_name}")
    print(f"Resolution: {intrinsics.width}x{intrinsics.height}")
    print(f"Focal Length (fx, fy): {intrinsics.fx:.2f}, {intrinsics.fy:.2f}")
    print(f"Principal Point (cx, cy): {intrinsics.ppx:.2f}, {intrinsics.ppy:.2f}")
    print(f"Distortion Model: {intrinsics.model}")
    print(f"Distortion Coefficients: {distortion_coeffs}")
    print("\nIntrinsic Matrix (K):")
    print(K_matrix)

    # Save this matrix to use in your Hand-Eye calibration script
    np.save("realsense_intrinsics.npy", K_matrix)
    print("\nSaved to 'realsense_intrinsics.npy'")

except RuntimeError as exc:
    raise RuntimeError(
        "Failed to start the RealSense stream inside Docker. If the camera is physically connected, "
        "make sure the container has access to the device USB bus, not only /dev/video nodes."
    ) from exc
finally:
    if pipeline_started:
        pipeline.stop()

RuntimeError: No RealSense device is visible to librealsense in this Docker container. If the camera is physically connected, recreate or reconfigure the container with USB passthrough (for example /dev/bus/usb and udev metadata), then rerun this cell.